# Análisis Inflación vs Tasas de interes Colombia

In [139]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import statsmodels 

import plotly.express as px
import plotly.graph_objects as go

inflacion = './data/inflacion.csv'
tasas_interes = './data/tasa_interes.csv'

height = 800

In [140]:
df_inflacion = pd.read_csv(inflacion, sep=',', encoding='utf-8')
df_tasasInteres = pd.read_csv(tasas_interes, sep=',', encoding='utf-8')

In [141]:
df_inflacion.columns = ['periodo', 'meta_inflacion', 'inflacion']
df_tasasInteres.columns = ['periodo', 'tasa_interes']

In [142]:
df_tasasInteres['periodo'] = pd.to_datetime(df_tasasInteres['periodo'], format='%Y/%m/%d')
df_inflacion['periodo'] = pd.to_datetime(df_inflacion['periodo'], format='%Y/%m/%d')

In [143]:
df_tasasInteres = df_tasasInteres[df_tasasInteres['periodo'] >= '2010-01-01']
df_inflacion = df_inflacion[df_inflacion['periodo'] >= '2010-01-01']

In [144]:
df_tasasInteres = df_tasasInteres.loc[df_tasasInteres.groupby(df_tasasInteres['periodo'].dt.to_period("M"))['periodo'].idxmax()]

In [145]:
df_inflacion

,periodo,meta_inflacion,inflacion
654,2010-01-31,3.0,2.10
655,2010-02-28,3.0,2.09
656,2010-03-31,3.0,1.84
657,2010-04-30,3.0,1.98
658,2010-05-31,3.0,2.07
...,...,...,...
831,2024-10-31,3.0,5.41
832,2024-11-30,3.0,5.20
833,2024-12-31,3.0,5.20
834,2025-01-31,NaN,5.22


In [146]:
df_data = pd.merge(df_inflacion, 
                   df_tasasInteres, 
                   on='periodo', 
                   how='left')

In [147]:
df_data['dif_inflacion_tasa'] = df_data['tasa_interes'] - df_data['inflacion']

In [148]:
df_data

,periodo,meta_inflacion,inflacion,tasa_interes,dif_inflacion_tasa
0,2010-01-31,3.0,2.10,3.50,1.40
1,2010-02-28,3.0,2.09,3.50,1.41
2,2010-03-31,3.0,1.84,3.50,1.66
3,2010-04-30,3.0,1.98,3.50,1.52
4,2010-05-31,3.0,2.07,3.00,0.93
...,...,...,...,...,...
177,2024-10-31,3.0,5.41,10.25,4.84
178,2024-11-30,3.0,5.20,9.75,4.55
179,2024-12-31,3.0,5.20,9.50,4.30
180,2025-01-31,NaN,5.22,9.50,4.28


In [149]:
import plotly.express as px
import plotly.graph_objects as go

# Crear la figura base con la inflación
fig = px.line(df_data,
              x='periodo',
              y='inflacion',
              markers=True,
              labels={'inflacion': 'Inflación'})

# Agregar la tasa de interés
fig.add_trace(go.Scatter(
    x=df_data['periodo'],
    y=df_data['tasa_interes'],
    mode='lines+markers',
    name='Tasa de Interés',
    line=dict(dash='dot', color='green'),  # Línea punteada en verde
    marker=dict(symbol='square', size=6)
))

# Ajustar etiquetas y formato del eje X
fig.update_layout(
    title='Inflación, Meta de Inflación y Tasa de Interés',
    xaxis_title='Periodo',
    yaxis_title='Porcentaje',
    height=height,
    xaxis=dict(
        tickformat="%Y-%m",
        dtick="M3"
    ),
    legend=dict(title='Indicadores')  # Título para la leyenda
)

fig.show()


In [150]:
# Calcular la correlación de Pearson
correlation_value = df_data[['inflacion', 'tasa_interes']].corr(method='pearson').iloc[0, 1]


In [151]:
#Crear el gráfico de dispersión con línea de tendencia
fig = px.scatter(df_data, 
                 x="tasa_interes", 
                 y="inflacion", 
                 trendline="ols",  # Agregar línea de regresión
                 title=f"Relación entre Tasa de Interés e Inflación (Correlación: {correlation_value:.2f})",
                 labels={"tasa_interes": "Tasa de Interés", "inflacion": "Inflación"},
                 opacity=0.7)

# Personalizar el gráfico
fig.update_traces(marker=dict(size=6, color='blue', opacity=0.6))  # Puntos en azul
fig.update_layout(
    xaxis_title="Tasa de Interés",
    yaxis_title="Inflación",
    height=height
)

fig.show()


In [155]:
import plotly.graph_objects as go

fig = go.Figure()

# Línea de la diferencia (tasa_interes - inflacion)
fig.add_trace(go.Scatter(
    x=df_data["periodo"], 
    y=df_data["dif_inflacion_tasa"], 
    mode="lines+markers", 
    name="Diferencia (Tasa - Inflación)",
    line=dict(color='purple')
))

# Línea de referencia en 0 para ver cuándo la inflación supera la tasa de interés
fig.add_trace(go.Scatter(
    x=df_data["periodo"], 
    y=[0] * len(df_data),  # Línea horizontal en 0
    mode="lines",
    name="Referencia 0",
    line=dict(color='black', dash="dash")
))

# Configurar el diseño del gráfico
fig.update_layout(
    title="Diferencia entre Tasa de Interés e Inflación",
    xaxis_title="Periodo",
    yaxis_title="Diferencia (Puntos de Separación)",
    height=height,
    xaxis=dict(tickformat="%Y-%m", dtick="M3")  # Formato Año-Mes en eje X
)

fig.show()
